# Session 4 — Walk-Forward Cross-Validation
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Standard k-fold cross-validation is **wrong for time series**. It randomly shuffles observations — so training on 2023 data and testing on 2019 data is allowed, which is a massive look-ahead bias.

Time series data has **temporal structure**. The future cannot be used to predict the past.

Walk-forward validation is how professional quant researchers validate models. It's also how you'll be expected to describe your methodology in interviews.

> 🎯 Interview answer: *"I used walk-forward cross-validation with a 6-month training window, 1-month test window, and a 2-week embargo between them to prevent overlap contamination."*


---
## 1️⃣ Why k-Fold Fails for Time Series

```
k-Fold (WRONG for time series):
─────────────────────────────────────────────────────
Fold 1: TRAIN [Mar,Jun,Sep,Dec 2021] TEST [Jan 2020]
Fold 2: TRAIN [Jan,Jun,Sep,Dec 2021] TEST [Mar 2020]
         ↑ trained on data AFTER the test period
         ↑ uses future information to predict the past
─────────────────────────────────────────────────────

Walk-Forward (CORRECT):
─────────────────────────────────────────────────────
Fold 1: TRAIN [Jan-Jun 2020] → TEST [Jul 2020]
Fold 2: TRAIN [Jan-Sep 2020] → TEST [Oct 2020]
Fold 3: TRAIN [Jan-Dec 2020] → TEST [Jan 2021]
         ↑ training window always ends before test window
         ↑ respects temporal ordering
─────────────────────────────────────────────────────
```


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Build dataset (same as Session 3)
spy = yf.download('SPY', start='2015-01-01', auto_adjust=True)['Close'].squeeze()
log_ret = np.log(spy / spy.shift(1))

features = pd.DataFrame(index=log_ret.index)
features['ret_1d']  = log_ret.shift(1)
features['ret_5d']  = log_ret.shift(1).rolling(5).mean()
features['ret_20d'] = log_ret.shift(1).rolling(20).mean()
features['vol_20d'] = log_ret.shift(1).rolling(20).std()
features['vol_60d'] = log_ret.shift(1).rolling(60).std()

df = pd.concat([features, log_ret.rename('target')], axis=1).dropna()
X = df.drop('target', axis=1).values
y = df['target'].values

print(f"Dataset: {len(df)} observations")


In [ ]:
# Compare: k-Fold vs TimeSeriesSplit
n_splits = 5
pipeline = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])

# WRONG: Standard k-Fold
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
kf_scores = []
for train_idx, test_idx in kf.split(X):
    pipeline.fit(X[train_idx], y[train_idx])
    pred = pipeline.predict(X[test_idx])
    kf_scores.append(r2_score(y[test_idx], pred))

# CORRECT: TimeSeriesSplit
tss = TimeSeriesSplit(n_splits=n_splits)
tss_scores = []
for train_idx, test_idx in tss.split(X):
    pipeline.fit(X[train_idx], y[train_idx])
    pred = pipeline.predict(X[test_idx])
    tss_scores.append(r2_score(y[test_idx], pred))

print(f"{'Method':<25} {'Mean R²':>10} {'Std R²':>10}")
print("-" * 47)
print(f"{'k-Fold (WRONG)':<25} {np.mean(kf_scores):>10.6f} {np.std(kf_scores):>10.6f}")
print(f"{'TimeSeriesSplit (CORRECT)':<25} {np.mean(tss_scores):>10.6f} {np.std(tss_scores):>10.6f}")
print()
print("k-Fold typically shows BETTER (overstated) R² because it trains on future data.")
print("TimeSeriesSplit gives the realistic out-of-sample estimate.")


---
## 2️⃣ Purging & Embargo — The Advanced Refinements

Even with TimeSeriesSplit, there are two remaining leakage sources:

**Purging:** Remove training samples that OVERLAP in time with the test set.
- If a feature uses a 20-day rolling window and the test starts Jan 21, training samples from Jan 1-20 contain overlapping data.
- Fix: remove the last `lookback_period` observations from training before each fold.

**Embargo:** Leave a gap between training and test.
- Returns on adjacent days are autocorrelated — day T+1 "knows about" day T.
- Fix: leave an N-day gap between the last training observation and first test observation.


In [ ]:
# Custom walk-forward with purging and embargo
def walk_forward_cv(X, y, n_splits=5, embargo_days=10, min_train_size=252):
    # Returns list of (train_indices, test_indices) tuples
    n = len(X)
    fold_size = (n - min_train_size) // n_splits
    splits = []

    for i in range(n_splits):
        test_start = min_train_size + i * fold_size
        test_end   = test_start + fold_size if i < n_splits - 1 else n

        # Embargo: remove last embargo_days from training
        train_end = test_start - embargo_days

        if train_end < min_train_size:
            continue

        train_idx = np.arange(0, train_end)
        test_idx  = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))

    return splits

# Run walk-forward with embargo
splits = walk_forward_cv(X, y, n_splits=5, embargo_days=10)
wf_scores = []
fold_info = []

pipeline = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])

for i, (train_idx, test_idx) in enumerate(splits):
    pipeline.fit(X[train_idx], y[train_idx])
    pred = pipeline.predict(X[test_idx])
    r2 = r2_score(y[test_idx], pred)
    wf_scores.append(r2)

    fold_info.append({
        'Fold': i+1,
        'Train obs': len(train_idx),
        'Test obs': len(test_idx),
        'Test R²': round(r2, 6)
    })

print("Walk-Forward with 10-day Embargo:")
print(pd.DataFrame(fold_info).to_string(index=False))
print(f"\nMean R²: {np.mean(wf_scores):.6f}")
print(f"Std R²:  {np.std(wf_scores):.6f}")


---
## 3️⃣ Evaluating a Trading Model — Beyond R²

R² near zero is **normal and expected** for return prediction models. The bar isn't predictive accuracy — it's economic significance.

| Metric | Formula | What's "good"? |
|--------|---------|---------------|
| Information Coefficient (IC) | Spearman correlation of predicted vs realised | IC > 0.05 consistently is useful |
| IC IR (Information Ratio of IC) | mean(IC) / std(IC) | IC IR > 0.5 suggests skill |
| Hit Rate | % of predictions with correct sign | > 52% is meaningful |
| Annual Turnover | How often positions change | Context-dependent |


In [ ]:
# Compute Information Coefficient across walk-forward folds
from scipy.stats import spearmanr

ic_scores = []
hit_rates = []

for train_idx, test_idx in splits:
    pipeline.fit(X[train_idx], y[train_idx])
    y_pred = pipeline.predict(X[test_idx])
    y_true = y[test_idx]

    # IC: Spearman rank correlation of predictions vs outcomes
    ic, _ = spearmanr(y_pred, y_true)
    ic_scores.append(ic)

    # Hit rate: did we get the sign right?
    correct_sign = np.sign(y_pred) == np.sign(y_true)
    hit_rates.append(correct_sign.mean())

ic_arr = np.array(ic_scores)
hr_arr = np.array(hit_rates)

print("=== Model Evaluation: Economic Significance ===")
print()
print(f"Information Coefficient (IC):")
print(f"  Mean IC:  {ic_arr.mean():.4f}  (>0.05 = useful)")
print(f"  IC Std:   {ic_arr.std():.4f}")
print(f"  IC IR:    {ic_arr.mean()/ic_arr.std():.4f}  (>0.5 = consistent skill)")
print()
print(f"Hit Rate (correct direction):")
print(f"  Mean:     {hr_arr.mean()*100:.2f}%  (>52% = meaningful)")
print(f"  Range:    {hr_arr.min()*100:.2f}% — {hr_arr.max()*100:.2f}%")
print()
print("Note: IC near 0 and hit rate near 50% is expected.")
print("Even a Sharpe 1.0 live strategy often has IC ~ 0.03-0.06.")


---
## ✅ Self-Check Questions

1. Why is k-fold cross-validation wrong for time series data?
   > *It randomly shuffles observations, allowing training on future data and testing on past data — a direct form of look-ahead bias.*

2. What is the embargo in walk-forward validation?
   > *A gap between the last training observation and the first test observation. Prevents leakage via autocorrelated returns on adjacent days.*

3. What is the Information Coefficient (IC)?
   > *Spearman rank correlation between model predictions and realised returns. Measures whether the model correctly ranks outcomes. IC > 0.05 consistently is considered useful.*

4. Why is near-zero R² acceptable for a return prediction model?
   > *Returns are close to random walks. A model with IC = 0.05 still has significant economic value despite explaining almost no variance. The bar is IC IR > 0.5, not R² > 0.1.*

---
## 🧪 Optional Tasks

- Visualise the TimeSeriesSplit folds. Plot training and test windows for each fold on a timeline.
- Implement a version where the training window is FIXED size (rolling window), not expanding. Compare performance.
- Compute IC per fold for your model. Is the IC stable across time, or does it degrade in recent folds? (IC degradation = regime change.)
- Test: what happens when you set embargo_days=0? Does performance change?
